# PlanIA P2 v5 · SDXL + LoRA de interiores + Qwen3-VL

SDXL genera en `cuda:0` con uno de los cinco LoRA del dataset `jeffreyamc/plania-p2-jjs-luxury-bedroom-xl`. Qwen3-VL describe el resultado en `cuda:1`.

En Kaggle, habilita **Internet** y **GPU T4 ×2**, adjunta la última versión del dataset y autoriza los secretos **HF_TOKEN** y **NGROK_TOKEN** para este notebook. Ejecuta las celdas en orden. Si reinstalas paquetes ya importados, reinicia el kernel antes de continuar.

Escoge el archivo en `lora_filename`, en la sección 3.1. El valor inicial es `Luxury.safetensors`.

La caché de modelos está en `/root/.cache/huggingface` y dura durante la sesión. El servidor se mantiene activo en la última celda. La URL pública y su ruta `/docs` aparecen en el log.

## 1. Instalar dependencias

PEFT permite cargar los LoRA en Diffusers.


In [ ]:
%pip install -q diffusers==0.35.2 transformers==4.57.1 accelerate==1.11.0 peft==0.17.1 fastapi uvicorn python-multipart pyngrok

## 2. Leer el secreto y seleccionar las GPUs

Validamos la configuraciÃ³n antes de descargar los pesos.

In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

import torch
import transformers
import diffusers
from importlib.metadata import version
from PIL import Image, ImageOps
from diffusers import StableDiffusionXLImg2ImgPipeline
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

print("En disco:", version("diffusers"), version("transformers"))
print("En memoria:", diffusers.__version__, transformers.__version__)

ngrok_token = secrets.get_secret("NGROK_TOKEN")

if torch.cuda.device_count() < 2:
    raise RuntimeError("Selecciona GPU T4—2 en Accelerator.")
device = torch.device("cuda:0")
qwen_device = torch.device("cuda:1")

cache_dir = "/root/.cache/huggingface"
Path(cache_dir).mkdir(parents=True, exist_ok=True)

print(device)
print(qwen_device)

## 3. Cargar SDXL en la primera T4

Usamos la revisión y el formato FP16 de la v3.

In [ ]:
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
model_revision = "462165984030d82259a11f4367a4eed129e94a7b"

pipeline = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    model_id,
    revision=model_revision,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
    cache_dir=cache_dir
)
pipeline.to(device)
pipeline.set_progress_bar_config(disable=True)

## 3.1. Escoger y cargar un LoRA

Cambia únicamente `lora_filename` por uno de estos nombres exactos:

- `Attic.safetensors`
- `Cyberpunk.safetensors`
- `Japanese.safetensors`
- `Luxury.safetensors`
- `NeoClassic.safetensors`

Los cinco archivos contienen LoRA para SDXL Base 1.0 según sus metadatos. El notebook carga solo el seleccionado y lo busca dentro de `/kaggle/input`. Adjunta la última versión del dataset para disponer de los cinco archivos.

`lora_scale = 0.8` controla la intensidad del adaptador. Para cambiar de estilo con el servidor activo, detén la última celda, cambia la variable, ejecuta esta celda y vuelve a iniciar el servidor. La celda descarga el adaptador anterior de la pipeline antes de cargar el nuevo.

Adapta el prompt enviado a `/generate` al estilo elegido. Para Luxury, incluye el activador **Luxury Bedroom**, por ejemplo:

> Luxury Bedroom, photorealistic interior, beige upholstered headboard, ivory linens, walnut panels, brass sconces, warm indirect lighting.

La transformación usa `strength=0.75`, 35 pasos y `guidance_scale=7.5`.


In [ ]:
# Opciones: Attic.safetensors, Cyberpunk.safetensors, Japanese.safetensors,
# Luxury.safetensors, NeoClassic.safetensors.
lora_filename = "Luxury.safetensors"
lora_scale = 0.8

lora_options = (
    "Attic.safetensors", "Cyberpunk.safetensors", "Japanese.safetensors",
    "Luxury.safetensors", "NeoClassic.safetensors",
)
if lora_filename not in lora_options:
    raise ValueError(f"LoRA no válido. Escoge uno de: {', '.join(lora_options)}")

lora_path = next(Path("/kaggle/input").rglob(lora_filename), None)
if lora_path is None:
    raise FileNotFoundError(
        f"No se encontró {lora_filename}. Adjunta la última versión de "
        "jeffreyamc/plania-p2-jjs-luxury-bedroom-xl en Input."
    )

pipeline.unload_lora_weights()
pipeline.load_lora_weights(
    str(lora_path.parent),
    weight_name=lora_path.name,
    adapter_name="selected_style",
    local_files_only=True,
)
pipeline.set_adapters("selected_style", adapter_weights=lora_scale)
print(f"LoRA cargado: {lora_path.name} | escala: {lora_scale}")


## 4. Cargar Qwen3-VL en la segunda T4

El modelo visual se mantiene en `cuda:1` durante toda la sesión.

In [ ]:
qwen_id = "Qwen/Qwen3-VL-4B-Instruct"
qwen_revision = "ebb281ec70b05090aa6165b016eac8ec08e71b17"
processor = AutoProcessor.from_pretrained(qwen_id, revision=qwen_revision, cache_dir=cache_dir)

qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    qwen_id,
    revision=qwen_revision,
    dtype=torch.float16,
    device_map={"": qwen_device.index},
    attn_implementation="sdpa",
    cache_dir=cache_dir
).eval()

## 5. Definir la API

`POST /generate` recibe los campos `image` y `prompt` como formulario multipart. Devuelve un JSON con `image_base64` (PNG), `image_mime_type` y `description`. El límite de entrada es 10 MiB y la API procesa una generación a la vez. `GET /health` indica que el servidor está listo.


### Ver los pesos descargados

Esta celda muestra los archivos de pesos y su tamaño dentro de la caché temporal.


In [ ]:
from pathlib import Path

cache_dir = Path("/root/.cache/huggingface")

weights = [
    path for path in cache_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".safetensors", ".bin", ".pt", ".pth"}
]

if weights:
    for path in sorted(weights):
        size_gib = path.stat().st_size / (1024 ** 3)
        print(f"{size_gib:6.2f} GiB  {path}")
else:
    print(f"No encontré pesos en {cache_dir}")

In [ ]:
import base64
from io import BytesIO
from threading import Lock

from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from PIL import UnidentifiedImageError

app = FastAPI(title="PlanIA P2 Â· SDXL y Qwen")
inference_lock = Lock()
max_upload_bytes = 10 * 1024 * 1024

@app.get("/health")
def health():
    return {"status": "ready"}

@app.post("/generate")
def generate(image: UploadFile = File(...), prompt: str = Form(...)):
    prompt = prompt.strip()
    if not prompt or len(prompt) > 2000:
        raise HTTPException(422, "El prompt debe tener entre 1 y 2000 caracteres.")
    data = image.file.read(max_upload_bytes + 1)
    if len(data) > max_upload_bytes:
        raise HTTPException(413, "La imagen supera 10 MiB.")
    try:
        with Image.open(BytesIO(data)) as source:
            if source.width * source.height > 25_000_000:
                raise HTTPException(413, "La imagen supera 25 megapíxeles.")
            reference = ImageOps.exif_transpose(source).convert("RGB")
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError):
        raise HTTPException(422, "No se pudo leer la imagen.") from None

    if not inference_lock.acquire(blocking=False):
        raise HTTPException(429, "Hay una generación en curso. Intenta nuevamente.")
    try:
        size = (768, 1024) if reference.height > reference.width else (1024, 768)
        prepared = ImageOps.pad(reference, size, Image.Resampling.LANCZOS)
        with torch.inference_mode():
            result = pipeline(
                prompt=prompt, image=prepared, strength=0.75,
                num_inference_steps=35, guidance_scale=7.5,
            ).images[0]
            messages = [{"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": (
                    "Describe en español esta imagen como especialista en fotografía "
                    "y diseño visual. Explica la composición, la iluminación, la "
                    "paleta y las texturas. Menciona solo detalles visibles."
                )},
            ]}]
            chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            preview = ImageOps.contain(result, (768, 768), Image.Resampling.LANCZOS)
            inputs = processor(text=[chat], images=[preview], return_tensors="pt")
            inputs.pop("token_type_ids", None)
            inputs = inputs.to(qwen_device, dtype=torch.float16)
            output = qwen_model.generate(**inputs, max_new_tokens=320, do_sample=False)
            tokens = output[:, inputs["input_ids"].shape[1]:]
            description = processor.batch_decode(tokens, skip_special_tokens=True)[0].strip()
        if not description:
            raise HTTPException(502, "Qwen devolvió una descripción vacía.")
        buffer = BytesIO()
        result.save(buffer, format="PNG")
        return {
            "image_base64": base64.b64encode(buffer.getvalue()).decode("ascii"),
            "image_mime_type": "image/png",
            "description": description,
        }
    finally:
        inference_lock.release()

## 6. Abrir el túnel y mantener el servidor activo

El agente de ngrok usa el authtoken de tu cuenta. Esta celda permanece en ejecución para mantener el puerto 8000 disponible. Al detener la sesión, el enlace deja de responder.


In [ ]:
from contextlib import asynccontextmanager

import uvicorn
from pyngrok import conf, ngrok

@asynccontextmanager
async def tunnel_lifespan(app):
    conf.get_default().auth_token = ngrok_token
    tunnel = ngrok.connect(addr=8000, proto="http")
    print(f"API pública: {tunnel.public_url}", flush=True)
    print(f"Formulario: {tunnel.public_url}/docs", flush=True)
    try:
        yield
    finally:
        ngrok.disconnect(tunnel.public_url)
        ngrok.kill()

app.router.lifespan_context = tunnel_lifespan

# Uvicorn comparte el bucle de eventos del notebook.
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000))
await server.serve()